### **Setup: Imports und Initialisierung**

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from quinex_application.quinex import Quinex
from quinex_application.visualisation_metadata import VisualisationMetadata
from quinex_application.mongos_store import MongoStore

In [ ]:
quinex = Quinex()
visualisation = VisualisationMetadata("../data/paper/paper_dataset.parquet")
mongodb = MongoStore(quinex, database_name="quinex_db_query4", collection_name="papers_2025")

### **Create Bulk Analysis**

In [ ]:
name = "quinex_application_query4"
response = quinex.intialize_analysis(name)
print(response)

### **Import and Sort Paper by Year**

In [ ]:
visualisation.load_df( query='Query: TITLE-ABS-KEY ( ( "energy system*" ) AND ( simul* OR model* OR "plan" OR "plans" OR "planning" OR optimi* OR analy* OR assess* OR evaluat* ) ) AND ( LIMIT-TO ( OA , "all" ) ) AND ( LIMIT-TO ( LANGUAGE , "English" ) )', date="14.05.2025")
df_doi = visualisation.df

In [ ]:
# Sort by year
dfs_by_year = {
    year: group_df
    for year, group_df in df_doi.groupby("Year")
}
df_2025 = dfs_by_year[2025]
print(len(df_2025))

### **Add Papers via DOI**

In [ ]:
mongodb.insert_paper(name, df_2025)

In [ ]:
collections = mongodb.get_collection("papers_2025")
print(len(collections))
print(collections[0].keys())

### **Successfully Added Papers**

In [ ]:
dois_df = mongodb.get_papers_from_collection(df_doi, "papers_2025")
display(dois_df)

### **Perform Bulk Analysis**

In [ ]:
dois = [doi for doi in dois_df["DOI"]]
mongodb.reset_quinex_result(dois)
mongodb.process_paper(name, dois)

### **Add Bulk Analysis to MongoDB**

In [ ]:
dois = [doi for doi in dois_df["DOI"]]
mongodb.add_analysis(name, dois)

## **Update Paper Parquet**

In [ ]:
df = mongodb.update_paper_parquet("papers_2025", "../data/paper/paper_dataset.parquet", "14.05.2025", "17.04.2025")

In [ ]:
display(df[df["Quinex"] == True])
display(df.tail(10))